# Task 3.2: MLP Baseline – Potato Quality Classification

This notebook establishes the **Basic MLP Baseline** for the potato quality classification project (fresh vs. rotten).

**Baselines covered:**
- Random Guess Accuracy
- Basic MLP Accuracy (with early stopping and learning rate tuning)

## 1. Imports

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

print('TensorFlow version:', tf.__version__)

## 2. Load Dataset

In [ ]:
# Paths (relative from this notebook's location)
BASE_PATH = '../Task 3.1 Create an Image Dataset ILO 6.2 a en b/data/processed'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')
VAL_PATH   = os.path.join(BASE_PATH, 'val')
TEST_PATH  = os.path.join(BASE_PATH, 'test')

# Image settings
IMG_SIZE   = (64, 64)   # resize all images to 64x64
BATCH_SIZE = 32

# Data generators (only rescaling, no augmentation for baseline)
datagen = ImageDataGenerator(rescale=1./255)

train_gen = datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen = datagen.flow_from_directory(
    VAL_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = datagen.flow_from_directory(
    TEST_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print('Classes:', train_gen.class_indices)
print('Train samples:', train_gen.samples)
print('Val samples:  ', val_gen.samples)
print('Test samples: ', test_gen.samples)

## 3. Random Guess Baseline

In [ ]:
# Count samples per class
class_counts = {}
for cls in os.listdir(TRAIN_PATH):
    cls_path = os.path.join(TRAIN_PATH, cls)
    if os.path.isdir(cls_path):
        class_counts[cls] = len(os.listdir(cls_path))

print('Class distribution (train):', class_counts)

total = sum(class_counts.values())
n_classes = len(class_counts)

# Calculate random guess accuracy
random_guess_acc = sum(
    (count / total) ** 2 for count in class_counts.values()
) * 100

print(f'\nNumber of classes: {n_classes}')
print(f'Random Guess Accuracy: {random_guess_acc:.2f}%')

## 4. Build & Compile MLP

In [ ]:
def build_model(print_summary=False):
    model = Sequential([
        Flatten(input_shape=(64, 64, 3)),   # flatten 64x64x3 image to 12288
        Dense(256, activation='relu'),
        Dense(128, activation='relu'),
        Dense(2, activation='softmax')       # 2 classes: fresh, rotten
    ])
    if print_summary:
        model.summary()
    return model

def compile_model(model, lr):
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model(print_summary=True)
model = compile_model(model, lr=0.001)

## 5. Train with Early Stopping

In [ ]:
callback = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

H = model.fit(
    train_gen,
    epochs=50,
    validation_data=val_gen,
    callbacks=[callback],
    verbose=1
)

print(f'\nTraining stopped at epoch {len(H.history["loss"])}')

## 6. Learning Curves

In [ ]:
def loss_plotter(H):
    plt.figure(figsize=(8, 5))
    plt.plot(H.history['loss'], label='Train loss')
    plt.plot(H.history['val_loss'], label='Val loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Learning Curve – Loss')
    plt.legend()
    plt.show()

def accuracy_plotter(H):
    plt.figure(figsize=(8, 5))
    plt.plot(H.history['accuracy'], label='Train accuracy')
    plt.plot(H.history['val_accuracy'], label='Val accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Learning Curve – Accuracy')
    plt.legend()
    plt.show()

loss_plotter(H)
accuracy_plotter(H)

## 7. Learning Rate Comparison

In [ ]:
learning_rates = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]

histories = []
best_val_losses = []

for lr in learning_rates:
    print(f'Training with lr={lr}')
    model = build_model()
    model = compile_model(model, lr=lr)
    
    callback = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )
    
    H_lr = model.fit(
        train_gen,
        epochs=20,
        validation_data=val_gen,
        callbacks=[callback],
        verbose=0
    )
    
    best_val_losses.append(min(H_lr.history['val_loss']))
    histories.append(H_lr)
    print(f'  → stopped at epoch {len(H_lr.history["loss"])}, best val_loss: {min(H_lr.history["val_loss"]):.4f}')

In [ ]:
# Plot val_loss vs epochs for each learning rate
plt.figure(figsize=(10, 6))
for i, H_lr in enumerate(histories):
    plt.plot(H_lr.history['val_loss'], label=f'lr={learning_rates[i]}')
plt.xlabel('Epochs')
plt.ylabel('Val Loss')
plt.title('Validation Loss per Learning Rate')
plt.legend()
plt.show()

In [ ]:
# Plot best val_loss vs learning rate
plt.figure(figsize=(8, 5))
plt.plot(learning_rates, best_val_losses, marker='o')
plt.xscale('log')
plt.xlabel('Learning Rate')
plt.ylabel('Best Val Loss')
plt.title('Best Validation Loss per Learning Rate')
plt.show()

best_lr = learning_rates[np.argmin(best_val_losses)]
print(f'Best learning rate: {best_lr}')

## 8. Final Evaluation on Test Set

In [ ]:
# Train final model with best learning rate
print(f'Training final model with best lr={best_lr}')

final_model = build_model()
final_model = compile_model(final_model, lr=best_lr)

callback = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

final_model.fit(
    train_gen,
    epochs=50,
    validation_data=val_gen,
    callbacks=[callback],
    verbose=0
)

# Evaluate on test set
test_loss, test_acc = final_model.evaluate(test_gen, verbose=0)
print(f'\nRandom Guess Accuracy: {random_guess_acc:.2f}%')
print(f'MLP Baseline Accuracy: {test_acc * 100:.2f}%')